# Test `ch1_python_basic.py`

Notebook này test các hàm trong `ch1_python_basic.py`.

**Cách chạy:** chọn kernel Python của dự án (`.venv`), rồi *Run All*.

**Lưu ý:** notebook **không** `import ch1_python_basic`. Lý do: import sẽ chạy toàn bộ code cấp ngoài cùng của file
(các lệnh `print`, ghi file `out/tickets.json`, và `asyncio.run(main())`).
`asyncio.run()` báo lỗi trong Jupyter, vì Jupyter đã có sẵn một event loop đang chạy.

Thay vào đó, cell đầu tiên chỉ nạp **import và định nghĩa hàm** từ file. Nhờ vậy notebook test đúng code thật của bạn:
sửa file `.py` rồi chạy lại cell đầu tiên là có code mới.

## 0. Nạp các hàm từ file

In [3]:
import ast
import asyncio
import io
import json
import tempfile
import time
from contextlib import redirect_stdout
from pathlib import Path

SOURCE = Path("../ch1_python_basic.py")


def load_functions(path: Path) -> dict:
    """Chỉ chạy các dòng import và định nghĩa hàm, bỏ qua code cấp ngoài cùng."""
    tree = ast.parse(path.read_text(encoding="utf-8"))
    keep = (ast.Import, ast.ImportFrom, ast.FunctionDef, ast.AsyncFunctionDef)
    tree.body = [node for node in tree.body if isinstance(node, keep)]
    namespace: dict = {}
    exec(compile(tree, str(path), "exec"), namespace)
    return namespace


ns = load_functions(SOURCE)
count_by = ns["count_by"]
titles_with = ns["titles_with"]
show_errors = ns["show_errors"]
fake_call = ns["fake_call"]

print("Đã nạp:", [k for k, v in ns.items() if callable(v) and not k.startswith("_") and v.__module__ is None])

Đã nạp: ['count_by', 'fake_call', 'main', 'titles_with', 'show_errors']


## Helper `check`

`check` ghi PASS hoặc FAIL cho từng trường hợp test. Một test FAIL hoặc báo lỗi **không** dừng cả notebook,
nên bạn thấy được hết kết quả trong một lần chạy.

In [ ]:
results: list[tuple[str, bool]] = []


def check(name: str, func, expected):
    """Chạy func(); so sánh kết quả với expected.

    expected có thể là một giá trị, hoặc một class Exception (ví dụ KeyError)
    khi bạn muốn test hàm có ném lỗi đó hay không.
    """
    try:
        actual = func()
    except Exception as e:  # noqa: BLE001
        ok = isinstance(expected, type) and isinstance(e, expected)
        actual = f"{type(e).__name__}: {e}"
    else:
        ok = actual == expected
    results.append((name, ok))
    print(f"{'✅ PASS' if ok else '❌ FAIL'}  {name}")
    if not ok:
        print(f"         mong đợi: {expected!r}")
        print(f"         thực tế:  {actual!r}")

In [8]:
from typing import Literal

from pydantic import BaseModel, ValidationError


class TicketOut(BaseModel):
    category: Literal["bug", "feature_request", "question"]
    severity: Literal["low", "medium", "high", "critical"]
    module: str
    summary: str
try:
  TicketOut.model_validate({'category': 'bug', 'severity': 'urgent','module': 'login', 'summary': 'summary'})
  print('Hợp lệ')
except ValidationError as e:
  err = e.errors()[0]
  print('Lỗi ở trường', err['loc'], '->', err['msg'])

Lỗi ở trường ('severity',) -> Input should be 'low', 'medium', 'high' or 'critical'


## Dữ liệu mẫu

In [ ]:
tickets = [
    {"id": 1, "title": "Lỗi login", "category": "bug"},
    {"id": 2, "title": "Thêm nút Excel", "category": "feature"},
    {"id": 3, "title": "Login chậm", "category": "bug"},
]

# Dữ liệu "xấu": thiếu khóa, giá trị None, chuỗi rỗng
messy = [
    {"id": 1, "title": "Lỗi LOGIN", "category": "bug"},
    {"id": 2, "category": "feature"},                  # thiếu title
    {"id": 3, "title": None, "category": None},        # giá trị None
    {"id": 4, "title": "", "category": ""},            # chuỗi rỗng
]

## 1. `count_by`

In [ ]:
check("đếm theo category", lambda: count_by(tickets), {"bug": 2, "feature": 1})
check("đếm theo khóa khác (id)", lambda: count_by(tickets, "id"), {1: 1, 2: 1, 3: 1})
check("list rỗng", lambda: count_by([]), {})
check("khóa không tồn tại", lambda: count_by(tickets, "priority"), {})

# Trường hợp biên: bạn muốn None và "" được đếm hay bị bỏ qua?
# Kết quả mong đợi dưới đây giả định là BỎ QUA. Sửa lại nếu ý định của bạn khác.
check("bỏ qua None và chuỗi rỗng", lambda: count_by(messy), {"bug": 1, "feature": 1})

## 2. `titles_with`

In [ ]:
check("tìm 'login', không phân biệt hoa thường",
      lambda: titles_with(tickets, "login"), ["Lỗi login", "Login chậm"])
check("tìm 'LOGIN' viết hoa", lambda: titles_with(tickets, "LOGIN"), ["Lỗi login", "Login chậm"])
check("tìm một phần của từ", lambda: titles_with(tickets, "exc"), ["Thêm nút Excel"])
check("không có kết quả", lambda: titles_with(tickets, "payment"), [])
check("list rỗng", lambda: titles_with([], "login"), [])

# Trường hợp biên với dữ liệu xấu
check("bỏ qua item thiếu title hoặc title None", lambda: titles_with(messy, "login"), ["Lỗi LOGIN"])
check("không khớp nhầm str(None) == 'None'", lambda: titles_with([{"title": None}], "none"), [])

## 3. `show_errors`

`show_errors` dùng `print` và không `return`, nên test bằng cách **bắt output** với `redirect_stdout`.

In [ ]:
def capture(func, *args) -> str:
    buf = io.StringIO()
    with redirect_stdout(buf):
        func(*args)
    return buf.getvalue()


check("in đúng số dòng", lambda: len(capture(show_errors, tickets).splitlines()), 3)
check("dòng đầu tiên", lambda: capture(show_errors, tickets).splitlines()[0], "Lỗi # 1: Lỗi login")
check("list rỗng không in gì", lambda: capture(show_errors, []), "")
check("trả về None", lambda: show_errors([]), None)
check("item thiếu title không báo lỗi", lambda: len(capture(show_errors, messy).splitlines()), 4)

## 4. `fake_call` (async)

Jupyter đã có event loop chạy sẵn, nên dùng **`await` trực tiếp** trong cell.
Không dùng `asyncio.run(...)` ở đây.

In [ ]:
r = await fake_call(5)
check("fake_call trả đúng chuỗi", lambda: r, "Kết quả 5")

start = time.perf_counter()
rs = await asyncio.gather(*(fake_call(i) for i in range(10)))
elapsed = time.perf_counter() - start

check("gather giữ đúng thứ tự", lambda: rs, [f"Kết quả {i}" for i in range(10)])
check("10 lời gọi chạy song song, tổng < 1.5s", lambda: elapsed < 1.5, True)
print(f"         thời gian: {elapsed:.2f}s")

## 5. Ghi và đọc JSON

Dùng thư mục tạm để không ghi đè `out/tickets.json` của bạn.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "tickets.json"
    path.write_text(json.dumps(tickets, ensure_ascii=False, indent=2), encoding="utf-8")
    loaded = json.loads(path.read_text(encoding="utf-8"))
    raw = path.read_text(encoding="utf-8")

check("đọc lại giống dữ liệu gốc", lambda: loaded, tickets)
check("giữ nguyên tiếng Việt (ensure_ascii=False)", lambda: "Lỗi login" in raw, True)
check("json.load với chuỗi báo lỗi", lambda: json.load("[]"), AttributeError)

## Tổng kết

In [ ]:
passed = sum(ok for _, ok in results)
print(f"{passed}/{len(results)} test PASS")
for name, ok in results:
    if not ok:
        print(f"❌ {name}")